# Scene 2 – Gemini Pipeline (Fixed + Debug Version)

## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image


## Debug Visualization

In [ ]:
def show_mask_debug(image_rgb, layer_name, anchor_mask, allowed_mask, forbidden_mask, focused_input):
    h, w = anchor_mask.shape
    comp = np.zeros((h, w, 3), dtype=np.uint8)

    comp[allowed_mask > 0] = [120, 200, 120]
    comp[forbidden_mask > 0] = [220, 80, 80]
    comp[anchor_mask > 0] = [80, 140, 255]

    overlap = (allowed_mask > 0) & (forbidden_mask > 0)
    comp[overlap] = [255, 255, 0]

    fig, axes = plt.subplots(2, 3, figsize=(16, 18))
    axes = axes.ravel()

    axes[0].imshow(image_rgb)
    axes[0].set_title(layer_name)
    axes[0].axis("off")

    axes[1].imshow(anchor_mask, cmap="gray")
    axes[1].set_title("anchor")

    axes[2].imshow(allowed_mask, cmap="gray")
    axes[2].set_title("allowed")

    axes[3].imshow(forbidden_mask, cmap="gray")
    axes[3].set_title("forbidden")

    axes[4].imshow(comp)
    axes[4].set_title("composite")

    axes[5].imshow(focused_input)
    axes[5].set_title("focused_input")

    plt.show()


## Mask Builder

In [ ]:
def build_control_masks_for_layer(layer_context, ordered_contexts):
    order = layer_context["order"]
    max_order = max(ctx["order"] for ctx in ordered_contexts)

    visible_mask = layer_context["visible_mask"] > 0
    ownership_mask = layer_context["ownership_mask"] > 0

    H, W = visible_mask.shape

    anchor_mask = visible_mask.copy()

    prev_visible = np.zeros((H, W), dtype=bool)
    future_visible = np.zeros((H, W), dtype=bool)

    for ctx in ordered_contexts:
        vm = ctx["visible_mask"] > 0
        if ctx["order"] < order:
            prev_visible |= vm
        elif ctx["order"] > order:
            future_visible |= vm

    forbidden_mask = prev_visible.copy()

    if order == 0:
        allowed_mask = np.ones((H, W), dtype=bool)
    elif order == max_order:
        allowed_mask = ownership_mask
    else:
        allowed_mask = np.logical_or(ownership_mask, future_visible)

    allowed_mask &= ~forbidden_mask
    allowed_mask |= anchor_mask

    return anchor_mask, allowed_mask, forbidden_mask


## Focus Image Builder

In [ ]:
def apply_focus_mask_fullframe(image_rgb, anchor_mask, allowed_mask, forbidden_mask):
    beige = np.full_like(image_rgb, 235)

    output = beige.copy()
    output[anchor_mask] = image_rgb[anchor_mask]
    output[forbidden_mask] = 0

    return output


## Debug Run (Before Gemini)

In [ ]:
ordered_contexts = sorted(layer_contexts, key=lambda x: x["order"])

for ctx in ordered_contexts:
    anchor_mask, allowed_mask, forbidden_mask = build_control_masks_for_layer(ctx, ordered_contexts)

    focused_input = apply_focus_mask_fullframe(
        image, anchor_mask, allowed_mask, forbidden_mask
    )

    show_mask_debug(
        image, ctx["layer_name"],
        anchor_mask, allowed_mask, forbidden_mask,
        focused_input
    )
